In [ ]:
# Setup: load the 8 table CSVs into a SQLite database inside Python, and set the chart style
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

tables_dir = "../data/clean"
table_names = ["directors", "actors", "genres", "certificates",
               "movies", "movie_genres", "movie_cast", "ratings"]

engine = sqlite3.connect(":memory:")
for name in table_names:
    pd.read_csv(f"{tables_dir}/{name}.csv").to_sql(name, engine, index=False)
    print("loaded", name)

HIGHLIGHT = "#004852"
BLUE = "#8598a7"
GREEN = "#678465"
GREY = "#c3c2b7"
TEXT = "#52514e"
TITLE = "#1f2124"

plt.rcParams.update({
    "figure.figsize": (9, 5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": TEXT, "xtick.color": TEXT,
    "ytick.color": TEXT, "axes.titleweight": "bold", "axes.titlesize": 13,
    "axes.grid": True, "grid.color": "#e5e4df", "grid.linewidth": 0.8, "axes.axisbelow": True, 
    "axes.titlecolor": TITLE, "figure.facecolor": "#f4f3ec", "axes.facecolor": "#f4f3ec", "savefig.facecolor": "#f4f3ec",
})

def hbar(data, label_col, value_col, title, xlabel, fmt="{:,.0f}", filename=None):
    """Horizontal bar chart, biggest on top, top bar highlighted, value at the end of each bar.
    If a filename is given, the chart is also saved as a PNG in the figures folder."""
    data = data.sort_values(value_col)
    colors = [HIGHLIGHT if v == data[value_col].max() else BLUE for v in data[value_col]]
    fig, ax = plt.subplots()
    bars = ax.barh(data[label_col], data[value_col], color=colors, height=0.6)
    ax.bar_label(bars, labels=[fmt.format(v) for v in data[value_col]], padding=4, color=TEXT, fontsize=9)
    ax.set_title(title, loc="left")
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", visible=False)
    ax.margins(x=0.12)
    plt.tight_layout()
    if filename:
        plt.savefig(f"../figures/{filename}", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
# Rating vs. box office queries

In [ ]:
# Average box office by IMDb rating group
q1_rating = pd.read_sql("""
SELECT
    CASE
        WHEN r.imdb_rating >= 8.5 THEN '8.5+'
        WHEN r.imdb_rating >= 8.0 THEN '8.0 - 8.4'
        ELSE '7.6 - 7.9'
    END AS rating_group,
    COUNT(*) AS films,
    ROUND(AVG(m.gross_revenue) / 1000000, 1) AS avg_gross_millions
FROM movies m
JOIN ratings r ON m.movie_id = r.movie_id
WHERE m.gross_revenue IS NOT NULL
GROUP BY rating_group
ORDER BY rating_group
""", engine)

# Average box office by number of votes group
q1_votes = pd.read_sql("""
SELECT
    CASE
        WHEN r.number_of_votes >= 1000000 THEN '1M+'
        WHEN r.number_of_votes >= 500000 THEN '500k - 1M'
        WHEN r.number_of_votes >= 100000 THEN '100k - 500k'
        ELSE 'under 100k'
    END AS votes_group,
    COUNT(*) AS films,
    ROUND(AVG(m.gross_revenue) / 1000000, 1) AS avg_gross_millions
FROM movies m
JOIN ratings r ON m.movie_id = r.movie_id
WHERE m.gross_revenue IS NOT NULL
GROUP BY votes_group
ORDER BY avg_gross_millions
""", engine)

In [ ]:
# Box office by rating group vs. by votes group, side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

bars1 = ax1.bar(q1_rating["rating_group"], q1_rating["avg_gross_millions"], color=BLUE, width=0.6)
ax1.bar_label(bars1, labels=[f"${v:,.0f}M" for v in q1_rating["avg_gross_millions"]], padding=3, color=TEXT, fontsize=9)
ax1.set_title("By IMDb rating", loc="left")
ax1.set_xlabel("IMDb rating")
ax1.set_ylabel("Average US box office (US$ millions)")
ax1.grid(axis="x", visible=False)

bars2 = ax2.bar(q1_votes["votes_group"], q1_votes["avg_gross_millions"], color=HIGHLIGHT, width=0.6)
ax2.bar_label(bars2, labels=[f"${v:,.0f}M" for v in q1_votes["avg_gross_millions"]], padding=3, color=TEXT, fontsize=9)
ax2.set_title("By number of votes (popularity)", loc="left")
ax2.set_xlabel("Number of IMDb votes")
ax2.grid(axis="x", visible=False)

fig.suptitle("Rating vs Popularity", x=0.01, ha="left",
             fontweight="bold", fontsize=14, color=TEXT)
plt.tight_layout()
plt.savefig("../figures/q2_rating_vs_votes.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# All films with box office data: rating and votes against box office
q1_films = pd.read_sql("""
SELECT
    m.movie_title,
    m.gross_revenue / 1000000.0 AS gross_millions,
    r.imdb_rating,
    r.number_of_votes
FROM movies m
JOIN ratings r ON m.movie_id = r.movie_id
WHERE m.gross_revenue IS NOT NULL
""", engine)

corr_rating = q1_films["imdb_rating"].corr(q1_films["gross_millions"])
corr_votes = q1_films["number_of_votes"].corr(q1_films["gross_millions"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

ax1.scatter(q1_films["imdb_rating"], q1_films["gross_millions"], s=18, color=BLUE, alpha=0.6, edgecolor="white", linewidth=0.4)
ax1.set_title(f"Rating vs box office (correlation {corr_rating:.2f})", loc="left", fontsize=11)
ax1.set_xlabel("IMDb rating")
ax1.set_ylabel("US box office (US$ millions)")

ax2.scatter(q1_films["number_of_votes"] / 1000000, q1_films["gross_millions"], s=18, color=HIGHLIGHT, alpha=0.5, edgecolor="white", linewidth=0.4)
ax2.set_title(f"Votes vs box office (correlation {corr_votes:.2f})", loc="left", fontsize=11)
ax2.set_xlabel("Number of IMDb votes (millions)")

plt.tight_layout()
plt.savefig("../figures/q2_scatter_rating_votes.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Top 10 highest-earning films

In [ ]:
# Top 10 highest-earning films, with rating and votes as labels
q1_top10 = pd.read_sql("""
SELECT
    m.movie_title,
    m.released_year,
    ROUND(m.gross_revenue / 1000000.0, 1) AS gross_millions,
    r.imdb_rating,
    r.number_of_votes
FROM movies m
JOIN ratings r ON m.movie_id = r.movie_id
WHERE m.gross_revenue IS NOT NULL
ORDER BY m.gross_revenue DESC
LIMIT 10
""", engine)

data = q1_top10.sort_values("gross_millions")
fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(data["movie_title"], data["gross_millions"], color=BLUE, height=0.6)
labels = [f"${g:,.0f}M  |  rating {r}  |  {v / 1000:,.0f}k votes"
          for g, r, v in zip(data["gross_millions"], data["imdb_rating"], data["number_of_votes"])]
ax.bar_label(bars, labels=labels, padding=4, color=TEXT, fontsize=8.5)
ax.set_title("Top 10 highest-earning films: average ratings, lots of votes", loc="left")
ax.set_xlabel("US box office (US$ millions)")
ax.grid(axis="y", visible=False)
ax.margins(x=0.55)
plt.tight_layout()
plt.savefig("../figures/q2_top10_earners.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Films per decade

In [ ]:
# Number of top films and average runtime per decade
q2_decades = pd.read_sql("""
SELECT
    (released_year / 10) * 10 AS decade,
    COUNT(*) AS films,
    ROUND(AVG(runtime)) AS avg_runtime_min
FROM movies
GROUP BY decade
ORDER BY decade
""", engine)

data = q2_decades[q2_decades["decade"] < 2020]
labels = data["decade"].astype(str) + "s"
colors = [HIGHLIGHT if d >= 2000 else BLUE for d in data["decade"]]

fig, ax = plt.subplots()
bars = ax.bar(labels, data["films"], color=colors, width=0.65)
ax.bar_label(bars, padding=3, color=TEXT, fontsize=9)
ax.set_title("Almost half of the top 1000 films are from the 2000s and 2010s", loc="left")
ax.set_ylabel("Number of films")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.savefig("../figures/q5_films_per_decade.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Runtime per decade

In [ ]:
# Average runtime per decade
fig, ax = plt.subplots()
ax.plot(labels, data["avg_runtime_min"], color=HIGHLIGHT, marker="o", linewidth=2)
for x, y in zip(labels, data["avg_runtime_min"]):
    ax.annotate(f"{y:.0f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", color=TEXT, fontsize=9)
ax.set_title("Top films grew to about 2 hours and stayed there", loc="left")
ax.set_ylabel("Average runtime (minutes)")
ax.set_ylim(70, 140)
plt.tight_layout()
plt.savefig("../figures/q5_runtime_per_decade.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# rising and falling genres

In [ ]:
# Rising and falling genres: % of films before 2000 vs. from 2000
q2_change = pd.read_sql("""
WITH before_2000 AS (
    SELECT g.genre_name,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM movies WHERE released_year < 2000), 1) AS pct_before_2000
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON mg.genre_id = g.genre_id
    WHERE m.released_year < 2000
    GROUP BY g.genre_name
),
from_2000 AS (
    SELECT g.genre_name,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM movies WHERE released_year >= 2000), 1) AS pct_from_2000
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON mg.genre_id = g.genre_id
    WHERE m.released_year >= 2000
    GROUP BY g.genre_name
)
SELECT b.genre_name,
       b.pct_before_2000,
       f.pct_from_2000,
       ROUND(f.pct_from_2000 - b.pct_before_2000, 1) AS change_pct_points
FROM before_2000 b
JOIN from_2000 f ON b.genre_name = f.genre_name
ORDER BY change_pct_points DESC
""", engine)

data = q2_change.sort_values("change_pct_points")
colors = [HIGHLIGHT if v > 0 else GREY for v in data["change_pct_points"]]

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(data["genre_name"], data["change_pct_points"], color=colors, height=0.6)
ax.bar_label(bars, labels=[f"{v:+.1f}" for v in data["change_pct_points"]], padding=4, color=TEXT, fontsize=9)
ax.axvline(0, color=TEXT, linewidth=0.8)
ax.set_title("Rising and falling genres (before 2000 vs. from 2000)", loc="left")
ax.set_xlabel("Change in % of top films (percentage points)")
ax.grid(axis="y", visible=False)
ax.margins(x=0.15)
plt.tight_layout()
plt.savefig("../figures/q5_genre_change.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Action, Adventure, War and Western as % of each decade's films
q2_genres_decade = pd.read_sql("""
SELECT
    (m.released_year / 10) * 10 AS decade,
    g.genre_name,
    COUNT(*) AS films
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
WHERE g.genre_name IN ('Action', 'Adventure', 'War', 'Western')
  AND m.released_year BETWEEN 1930 AND 2019
GROUP BY decade, g.genre_name
ORDER BY decade
""", engine)

# turn counts into % of each decade's films, so decades of different sizes can be compared
data = q2_genres_decade.merge(q2_decades[["decade", "films"]], on="decade", suffixes=("", "_total"))
data["pct"] = 100 * data["films"] / data["films_total"]
data = data.pivot(index="decade", columns="genre_name", values="pct").fillna(0)

genre_colors = {"Action": HIGHLIGHT, "Adventure": GREEN, "War": BLUE, "Western": GREY}
fig, ax = plt.subplots()
for genre, color in genre_colors.items():
    ax.plot(data.index.astype(str) + "s", data[genre], marker="o", linewidth=2, color=color, label=genre)
ax.set_title("Fastest-growing and fastest-declining genres, by decade", loc="left")
ax.set_ylabel("% of the decade's top films")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("../figures/q5_genres_per_decade.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Average vs. median box office by audience group (Unrated excluded: mostly old films with little box office data)
q3_films = pd.read_sql("""
SELECT c.audience, m.gross_revenue / 1000000.0 AS gross_millions
FROM movies m
JOIN certificates c ON m.certificate_id = c.certificate_id
WHERE m.gross_revenue IS NOT NULL
  AND c.audience <> 'Unrated'
""", engine)

data = (q3_films.groupby("audience")["gross_millions"]
        .agg(average="mean", median="median")
        .sort_values("average"))
y = range(len(data))

fig, ax = plt.subplots()
bars_avg = ax.barh([i + 0.2 for i in y], data["average"], height=0.4, color=HIGHLIGHT, label="Average")
bars_med = ax.barh([i - 0.2 for i in y], data["median"], height=0.4, color=BLUE, label="Median (typical film)")
ax.bar_label(bars_avg, labels=[f"${v:,.0f}M" for v in data["average"]], padding=3, color=TEXT, fontsize=9)
ax.bar_label(bars_med, labels=[f"${v:,.0f}M" for v in data["median"]], padding=3, color=TEXT, fontsize=9)
ax.set_yticks(list(y))
ax.set_yticklabels(data.index)
ax.set_title("Family/Teen films earn the most, on average and for a typical film", loc="left")
ax.set_xlabel("US box office per film (US$ millions)")
ax.grid(axis="y", visible=False)
ax.margins(x=0.15)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.savefig("../figures/q6_box_office_by_audience.png", dpi=200, bbox_inches="tight")
plt.show()